# SQLChat Backend Experimentation Notebook

This notebook replicates the core functionality of the `sqlchat-backend` project. It allows you to:
1. Generate SQL queries from natural language questions using Ollama.
2. Execute those queries against a PostgreSQL database.
3. Generate natural language responses based on the query results using Ollama.
4. Insert data into the database.
5. Experiment with different questions and prompts.

## 1. Import Libraries and Load Environment Variables
Import necessary libraries and load environment variables from the `.env` file. Make sure your `.env` file is present in the project root and contains `DATABASE_URL` and optionally `OLLAMA_HOST`.

In [ ]:
import os
import re
import requests
import sqlalchemy
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import json # For pretty printing results

# Load environment variables from .env file in the parent directory
dotenv_path = os.path.join(os.pardir, '.env') # Assumes notebook is in a subdir like 'notebooks' or project root
if not os.path.exists(dotenv_path):
    # If not found in parent, check current directory (if notebook is in root)
    dotenv_path = '.env'

load_dotenv(dotenv_path=dotenv_path)

DATABASE_URL = os.getenv("DATABASE_URL")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434") # Default if not set

print(f"Database URL: {DATABASE_URL}")
print(f"Ollama Host: {OLLAMA_HOST}")

# Basic check
if not DATABASE_URL:
    print("Error: DATABASE_URL not found in environment variables. Please check your .env file.")
if not OLLAMA_HOST:
     print("Error: OLLAMA_HOST not found. Using default http://localhost:11434")


## 2. Define Database Connection and Query Function
Set up the connection to your PostgreSQL database using SQLAlchemy and define a function to execute queries.

In [ ]:
# Ensure DATABASE_URL is loaded before creating the engine
if DATABASE_URL:
    try:
        engine = create_engine(DATABASE_URL)
        print("SQLAlchemy engine created successfully.")
    except Exception as e:
        print(f"Error creating SQLAlchemy engine: {e}")
        engine = None
else:
    print("Cannot create SQLAlchemy engine because DATABASE_URL is not set.")
    engine = None

def run_query(query: str):
    """Executes a SQL query and returns the result or an error."""
    if not engine:
        return {"error": "Database engine not initialized."}
    try:
        with engine.connect() as conn:
            # For SELECT statements, fetch results
            if query.strip().upper().startswith("SELECT") or query.strip().upper().startswith("WITH"):
                 result = conn.execute(text(query))
                 rows = result.fetchall()
                 # Close cursor/connection before returning data
                 conn.commit() # Although not strictly needed for SELECT, good practice
                 return [dict(row._mapping) for row in rows]
            # For INSERT, UPDATE, DELETE, execute and commit
            else:
                 conn.execute(text(query))
                 conn.commit()
                 return {"status": "Query executed successfully."}
    except Exception as e:
        # Attempt to rollback on error, though commit might have happened depending on error timing
        try:
            conn.rollback()
        except: # Handle cases where connection might already be closed or invalid
            pass
        print(f"Error executing query:\n{query}\nError: {e}")
        return {"error": str(e)}

# Test connection (optional)
# try:
#     with engine.connect() as connection:
#         print("Database connection successful.")
# except Exception as e:
#     print(f"Database connection failed: {e}")

## 3. Define Ollama Interaction Functions
Define functions to communicate with the Ollama API. One function generates SQL, and the other generates natural language responses.

In [ ]:
def ask_ollama_sql(prompt: str, model: str = "sqlcoder:15b"):
    """Ask Ollama to generate SQL and extract only the SQL part."""
    print(f"--- Sending SQL generation prompt to Ollama (model: {model}) ---")
    # print(f"Prompt:\n{prompt}") # Uncomment to debug the exact prompt
    try:
        response = requests.post(f"{OLLAMA_HOST}/api/generate", json={
            "model": model,
            "prompt": prompt,
            "stream": False
        })
        response.raise_for_status() # Raise an exception for bad status codes

        response_data = response.json()
        text = response_data.get("response", "").strip()
        print("--- Received response from Ollama ---")
        # print(f"Raw response:\n{text}") # Uncomment to see the full raw response

        # Remove markdown formatting
        text = text.replace("```sql", "").replace("```", "").strip()

        # Find the first valid SQL keyword and return from there
        lines = text.splitlines()
        sql_start_index = -1
        for i, line in enumerate(lines):
            if re.match(r"^\s*(SELECT|WITH|INSERT|UPDATE|DELETE)\s", line.strip(), re.IGNORECASE):
                sql_start_index = i
                break

        if sql_start_index != -1:
            extracted_sql = "\n".join(lines[sql_start_index:]).strip()
            print(f"--- Extracted SQL ---")
            # print(extracted_sql) # Print the final extracted SQL
            return extracted_sql
        else:
             print("--- ERROR: Could not find valid SQL starting keyword in response ---")
             print(f"Full response was:\n{text}")
             raise ValueError("Generated response did not contain valid SQL starting with SELECT, WITH, INSERT, UPDATE, or DELETE.")

    except requests.exceptions.RequestException as e:
        print(f"--- ERROR: Ollama API request failed: {e} ---")
        raise ConnectionError(f"Failed to connect to Ollama API at {OLLAMA_HOST}: {e}")
    except Exception as e:
        print(f"--- ERROR: An unexpected error occurred during Ollama SQL generation: {e} ---")
        raise


def ask_ollama_response(prompt: str, model: str = "llama3"):
    """Ask Ollama for a plain English response."""
    print(f"--- Sending response generation prompt to Ollama (model: {model}) ---")
    # print(f"Prompt:\n{prompt}") # Uncomment to debug the exact prompt
    try:
        response = requests.post(f"{OLLAMA_HOST}/api/generate", json={
            "model": model,
            "prompt": prompt,
            "stream": False
        })
        response.raise_for_status() # Raise an exception for bad status codes

        response_data = response.json()
        text = response_data.get("response", "").strip()
        print("--- Received response from Ollama ---")
        # print(f"Generated response:\n{text}") # Uncomment to see the raw response
        return text

    except requests.exceptions.RequestException as e:
        print(f"--- ERROR: Ollama API request failed: {e} ---")
        raise ConnectionError(f"Failed to connect to Ollama API at {OLLAMA_HOST}: {e}")
    except Exception as e:
        print(f"--- ERROR: An unexpected error occurred during Ollama response generation: {e} ---")
        raise

## 4. Define SQL and Response Generation Functions
Define the higher-level functions that construct the prompts and call the Ollama interaction functions.

In [ ]:
def generate_sql(schema: str, question: str, model: str = "sqlcoder:15b") -> str:
    """Generates SQL query from a question based on the schema."""
    # Note: Using a slightly simpler prompt than the systemprompt.txt for direct use here.
    # You could load the prompt from schema/systemprompt.txt if preferred.
    prompt = f"""
You are a PostgreSQL SQL expert.
Given the following database schema:
{schema}

Generate a valid PostgreSQL query to answer the question: "{question}"

Rules:
- Only return the SQL query.
- Do not include explanations, comments, or markdown formatting like ```sql.
- The query must start with SELECT, INSERT, UPDATE, DELETE, or WITH.
- Use EXTRACT(MONTH FROM column) and EXTRACT(YEAR FROM column) for date parts; do not use MONTH() or YEAR().
- Use table aliases if joining multiple tables.

SQL Query:
"""
    return ask_ollama_sql(prompt, model=model)


def generate_response(question: str, result: str, model: str = "llama3") -> str:
    """Generates a plain language response based on the question and SQL result."""
    prompt = f"""Based on the following question and SQL query result, provide a concise, plain language answer.

Original Question: {question}

SQL Query Result:
{result}

Answer the question based *only* on the provided result data.
Answer:"""
    # Limit result size to avoid overly long prompts (adjust limit as needed)
    max_result_length = 2000
    if len(result) > max_result_length:
        result = result[:max_result_length] + "\n... (result truncated)"

    return ask_ollama_response(prompt, model=model)

## 5. Set Database Schema
Define the database schema as a string. Replace this with your actual database schema.

In [ ]:
# Example schema (replace with your actual schema)
schema = """
CREATE TABLE trips (
    trip_id SERIAL PRIMARY KEY,
    member_id INTEGER,
    vehicle_id INTEGER,
    pickup_station VARCHAR(100),
    dropoff_station VARCHAR(100),
    pickup_time TIMESTAMP,
    dropoff_time TIMESTAMP,
    status VARCHAR(50),
    revenue DECIMAL(10, 2)
);

CREATE TABLE vehicles (
    vehicle_id SERIAL PRIMARY KEY,
    make VARCHAR(50),
    model VARCHAR(50),
    fuel_type VARCHAR(50),
    type VARCHAR(50),
    station_id INTEGER
);

CREATE TABLE members (
    member_id SERIAL PRIMARY KEY,
    full_name VARCHAR(100),
    email VARCHAR(100) UNIQUE,
    phone_number VARCHAR(20),
    status VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Example foreign key relationships (add actual constraints if known)
-- ALTER TABLE trips ADD CONSTRAINT fk_member FOREIGN KEY (member_id) REFERENCES members(member_id);
-- ALTER TABLE trips ADD CONSTRAINT fk_vehicle FOREIGN KEY (vehicle_id) REFERENCES vehicles(vehicle_id);
"""

print("Database schema defined.")
# print(schema)

## 6. Generate SQL from a Natural Language Question
Provide a question and use the `generate_sql` function to create the corresponding SQL query.

In [ ]:
# --- INPUT YOUR QUESTION HERE ---
question = "How many members are there in total?"
# question = "What are the different types of vehicles available?"
# question = "Show me the names and email addresses of members created this year."
# question = "What was the total revenue generated from trips in March 2024?"

print(f"Question: {question}\n")

try:
    generated_sql = generate_sql(schema, question)
    print("\n--- Generated SQL ---")
    print(generated_sql)
except Exception as e:
    print(f"\n--- Error generating SQL ---")
    print(e)
    generated_sql = None # Ensure variable exists but is None on error

## 7. Execute Generated SQL Query
Run the SQL query generated in the previous step against the database.

In [ ]:
sql_result = None # Initialize result variable

if generated_sql:
    print("\n--- Executing SQL ---")
    print(generated_sql)
    sql_result = run_query(generated_sql)

    print("\n--- Query Result ---")
    # Pretty print the result if it's a list (SELECT) or dict (status/error)
    if isinstance(sql_result, list) or isinstance(sql_result, dict):
         print(json.dumps(sql_result, indent=2, default=str)) # Use default=str for non-serializable types like Decimal
    else:
         print(sql_result) # Print as is if it's neither
else:
    print("\nSkipping execution because SQL generation failed or was skipped.")


## 8. Generate Natural Language Response from SQL Result
Use the original question and the result from the database query to generate a plain language answer.

In [ ]:
natural_response = None # Initialize response variable

if sql_result is not None and "error" not in sql_result:
    print(f"\n--- Generating Natural Language Response ---")
    print(f"Original Question: {question}")
    # Convert result to string for the prompt
    result_str = json.dumps(sql_result, default=str)
    print(f"SQL Result (as string input): {result_str}")

    try:
        natural_response = generate_response(question, result_str)
        print("\n--- Natural Language Response ---")
        print(natural_response)
    except Exception as e:
        print(f"\n--- Error generating response ---")
        print(e)

elif sql_result is not None and "error" in sql_result:
    print("\nSkipping natural language response generation due to SQL execution error.")
    natural_response = f"I couldn't answer the question because the generated SQL query resulted in an error: {sql_result.get('error', 'Unknown database error')}"
    print(f"\n--- Error Response ---")
    print(natural_response)
else:
    print("\nSkipping natural language response generation because SQL execution was skipped or failed.")
    natural_response = "I couldn't answer the question because there was an issue generating or executing the SQL query."
    print(f"\n--- Error Response ---")
    print(natural_response)

## 9. Insert Data into the Database
Demonstrate using the `run_query` function to execute `INSERT` statements.

**Caution:** Running INSERT/UPDATE/DELETE queries will modify your database. Make sure you are connected to the correct database and understand the query before executing it.

In [ ]:
# --- Example INSERT Query ---
# Modify this query according to your schema and the data you want to insert.
# Ensure data types match your table definition (e.g., strings in quotes, numbers without).

# Example: Insert a new member
insert_query = """
INSERT INTO members (full_name, email, phone_number, status)
VALUES ('Test User', 'test.user@example.com', '123-456-7890', 'active');
"""

# Example: Insert a new vehicle
# insert_query = """
# INSERT INTO vehicles (make, model, fuel_type, type, station_id)
# VALUES ('Toyota', 'Camry', 'Gasoline', 'Sedan', 1);
# """

print("--- Preparing INSERT Query ---")
print(insert_query)

# --- Uncomment the line below to execute the INSERT query ---
# insert_result = run_query(insert_query)

# --- Check the result ---
# print("\n--- INSERT Result ---")
# if 'insert_result' in locals():
#      print(json.dumps(insert_result, indent=2))
# else:
#      print("INSERT query was not executed (commented out).")

# You can verify the insertion by running a SELECT query afterwards
# verify_query = "SELECT * FROM members WHERE email = 'test.user@example.com';"
# verification_result = run_query(verify_query)
# print("\n--- Verification Result ---")
# print(json.dumps(verification_result, indent=2, default=str))

## 10. Experiment with Different Questions and Prompts
Modify the inputs below to see how changes affect the generated SQL and responses.

In [ ]:
# --- Experiment Area ---

# 1. Try a different question:
new_question = "Which vehicle type had the most trips?"
# new_question = "List members who haven't had any trips."
# new_question = "What is the average revenue per trip for electric vehicles?"

print(f"--- Experimenting with New Question ---")
print(f"Question: {new_question}\n")

try:
    exp_sql = generate_sql(schema, new_question)
    print("\n--- Generated SQL ---")
    print(exp_sql)

    if exp_sql:
        exp_result = run_query(exp_sql)
        print("\n--- Query Result ---")
        print(json.dumps(exp_result, indent=2, default=str))

        if exp_result is not None and "error" not in exp_result:
             exp_response = generate_response(new_question, json.dumps(exp_result, default=str))
             print("\n--- Natural Language Response ---")
             print(exp_response)
        elif exp_result is not None:
             print(f"\n--- SQL Error ---")
             print(exp_result.get('error'))
        else:
             print("\n--- Execution Skipped ---")

except Exception as e:
    print(f"\n--- Error during experiment ---")
    print(e)


In [ ]:
# 2. Modify the SQL Generation Prompt (Example: Add a rule)

def generate_sql_modified_prompt(schema: str, question: str, model: str = "sqlcoder:15b") -> str:
    """Generates SQL using a modified prompt."""
    prompt = f"""
You are a PostgreSQL SQL expert.
Given the following database schema:
{schema}

Generate a valid PostgreSQL query to answer the question: "{question}"

Rules:
- Only return the SQL query.
- Do not include explanations, comments, or markdown formatting like ```sql.
- The query must start with SELECT, INSERT, UPDATE, DELETE, or WITH.
- Use EXTRACT(MONTH FROM column) and EXTRACT(YEAR FROM column) for date parts.
- Use table aliases if joining multiple tables.
- **NEW RULE: Always order results by the primary key of the main table involved.**

SQL Query:
"""
    return ask_ollama_sql(prompt, model=model)

# --- Test the modified prompt ---
question_for_ordering = "List all vehicle makes and models."
print(f"--- Experimenting with Modified SQL Prompt ---")
print(f"Question: {question_for_ordering}\n")

try:
    mod_sql = generate_sql_modified_prompt(schema, question_for_ordering)
    print("\n--- Generated SQL (Modified Prompt) ---")
    print(mod_sql)
    # Optional: Execute and see if ordering is applied
    # mod_result = run_query(mod_sql)
    # print("\n--- Query Result (Modified Prompt) ---")
    # print(json.dumps(mod_result, indent=2, default=str))

except Exception as e:
    print(f"\n--- Error during modified prompt experiment ---")
    print(e)


In [ ]:
# 3. Modify the Response Generation Prompt (Example: Change tone)

def generate_response_modified_prompt(question: str, result: str, model: str = "llama3") -> str:
    """Generates a response using a modified prompt."""
    prompt = f"""You are a helpful assistant. You received the following data as a result of a database query.

Original Question: {question}

Data:
{result}

Explain the data clearly and concisely in response to the original question. Be friendly and start your response with "Okay, here's what I found:".
Answer:"""
    max_result_length = 2000
    if len(result) > max_result_length:
        result = result[:max_result_length] + "\n... (result truncated)"

    return ask_ollama_response(prompt, model=model)

# --- Test the modified response prompt (using previous results if available) ---
print(f"--- Experimenting with Modified Response Prompt ---")

# Use results from the first experiment if they exist and were successful
if 'exp_result' in locals() and exp_result is not None and "error" not in exp_result:
    print(f"Using results from question: '{new_question}'")
    try:
        mod_response = generate_response_modified_prompt(new_question, json.dumps(exp_result, default=str))
        print("\n--- Natural Language Response (Modified Prompt) ---")
        print(mod_response)
    except Exception as e:
        print(f"\n--- Error during modified response experiment ---")
        print(e)
else:
    print("Skipping modified response experiment as previous results are unavailable or had errors.")


End of Notebook.